# Dental Cavity Detection — U-Net Segmentation
### Notebook order of execution

1. **Setup** — mount Google Drive, install packages, print versions.
2. **Exploratory Data Analysis (EDA)** — run *before* training. Fully
   self-contained (all variables prefixed `eda_`); it only *reads* the dataset
   and does not touch the training code.
3. **Training** — the original pipeline, now split into small cells so each
   step (config → data → model → loss → compile → callbacks → fit → save)
   runs and reports on its own.
4. **Evaluation** — extra cells at the end that score the trained model on the
   validation set: Dice, IoU, precision, recall, F1, a threshold sweep,
   a pixel confusion matrix, the training curves, and sample predictions.

Run the cells top to bottom.


## 0 · Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:
!pip install -q tensorflow opencv-python matplotlib numpy


In [ ]:
import sys
import tensorflow as tf
import cv2
import numpy as np

print("Python version       :", sys.version)
print("TensorFlow version   :", tf.__version__)
print("Keras version        :", tf.keras.__version__)
print("OpenCV version       :", cv2.__version__)
print("NumPy version        :", np.__version__)


# ============================================================
# EXPLORATORY DATA ANALYSIS (EDA)  --  Dental Cavity Dataset
# ============================================================

**This section is fully self-contained.** It defines its own paths, imports and
variables (all prefixed `eda_`) and only *reads* the dataset, so it does not
change or depend on any of the training cells above. Run the Drive-mount cell
once, then run these cells top to bottom.

What we look at, and why it matters:

1. **Inventory & integrity** — how many image/mask pairs exist, are they all
   paired, same size, same format? (Catches silent data bugs before training.)
2. **Image intensity distribution** — how bright/contrasty the X-rays are.
   Justifies the `/255` normalization and hints whether contrast enhancement
   (e.g. CLAHE) could help.
3. **Class (im)balance** — what fraction of pixels are actually "cavity".
   This is the single most important plot: it is the reason the model uses
   **focal + dice loss** instead of plain cross-entropy.
4. **Lesion geometry** — how many cavities per patch and how large they are.
5. **Visual samples** — image, mask and overlay grids to eyeball label quality.


In [ ]:
# ---- EDA config & imports (standalone) ----------------------------------
import os, cv2, numpy as np
import matplotlib.pyplot as plt

# Same dataset location used by the training cells, but kept in local vars
EDA_BASE = "/content/drive/MyDrive/teeth/patch_dataset"
EDA_SPLITS = {
    "train": (f"{EDA_BASE}/train/images", f"{EDA_BASE}/train/masks"),
    "val":   (f"{EDA_BASE}/val/images",   f"{EDA_BASE}/val/masks"),
}
EDA_POS_THRESH = 127          # a mask pixel > this is treated as "cavity"
EDA_PALETTE = ["#1E3A8F", "#C4453B", "#8FAE9C"]   # navy / red / sage

plt.rcParams.update({
    "figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11,
})

def eda_list_pngs(d):
    """Sorted .png filenames in a folder (empty list if folder is missing)."""
    if not os.path.isdir(d):
        print("  [warn] folder not found:", d)
        return []
    return sorted(f for f in os.listdir(d) if f.lower().endswith(".png"))

print("EDA config ready. Splits:", list(EDA_SPLITS))


## 1 · Dataset inventory & integrity

We count images and masks per split and confirm every image has a mask with the
**exact same filename** (that is how this dataset pairs them). We also verify all
patches are the expected `128×128` size, list the pixel value range of the masks
(should be essentially `{0, 255}`), and check the train/val split has **no leaked
filenames** in common.


In [ ]:
# ---- 1. Inventory, pairing, sizes, split leakage ------------------------
eda_inventory = {}
eda_split_files = {}

for split, (img_dir, msk_dir) in EDA_SPLITS.items():
    imgs = eda_list_pngs(img_dir)
    msks = set(eda_list_pngs(msk_dir))
    paired  = [f for f in imgs if f in msks]
    img_only = [f for f in imgs if f not in msks]     # image without a mask
    msk_only = [f for f in msks if f not in set(imgs)] # mask without an image
    eda_inventory[split] = dict(images=len(imgs), masks=len(msks),
                                paired=len(paired),
                                images_without_mask=len(img_only),
                                masks_without_image=len(msk_only))
    eda_split_files[split] = set(imgs)

print("=" * 60)
print(f"{'split':<8}{'images':>9}{'masks':>9}{'paired':>9}{'img w/o msk':>13}{'msk w/o img':>13}")
print("-" * 60)
for s, v in eda_inventory.items():
    print(f"{s:<8}{v['images']:>9}{v['masks']:>9}{v['paired']:>9}"
          f"{v['images_without_mask']:>13}{v['masks_without_image']:>13}")
print("=" * 60)

# train / val filename leakage
if "train" in eda_split_files and "val" in eda_split_files:
    overlap = eda_split_files["train"] & eda_split_files["val"]
    print(f"Train/Val filename overlap (leakage): {len(overlap)}  "
          + ("OK - disjoint" if len(overlap) == 0 else "!! LEAKAGE !!"))

# size + mask value sanity check on a sample from each split
print("\nSize / dtype / mask-value check (sampled):")
for split, (img_dir, msk_dir) in EDA_SPLITS.items():
    files = eda_list_pngs(img_dir)
    if not files:
        continue
    sizes, mvals = set(), set()
    for f in files[:300]:
        im = cv2.imread(os.path.join(img_dir, f), cv2.IMREAD_GRAYSCALE)
        mk = cv2.imread(os.path.join(msk_dir, f), cv2.IMREAD_GRAYSCALE)
        if im is not None: sizes.add(im.shape)
        if mk is not None: mvals.update(np.unique(mk).tolist()[:5])
    print(f"  {split:<6} image sizes={sizes}  "
          f"mask unique-values(sample)={sorted(mvals)[:6]}{'...' if len(mvals)>6 else ''}")


## 2 · Compute per-image statistics

We scan every patch once and record, per image: its **mean intensity** (how
bright the X-ray is) and its **cavity-pixel fraction** (share of the mask that is
positive). The patches are tiny (`128×128`) so scanning all ~5,000 is fast. These
arrays feed all the plots below.


In [ ]:
# ---- 2. One pass over the data to collect statistics --------------------
eda_stats = {}   # split -> dict of numpy arrays

for split, (img_dir, msk_dir) in EDA_SPLITS.items():
    files = eda_list_pngs(img_dir)
    if not files:
        continue
    img_means, pos_frac, n_lesions, lesion_areas = [], [], [], []

    for f in files:
        mk = cv2.imread(os.path.join(msk_dir, f), cv2.IMREAD_GRAYSCALE)
        im = cv2.imread(os.path.join(img_dir, f), cv2.IMREAD_GRAYSCALE)
        if mk is None or im is None:
            continue
        binm = (mk > EDA_POS_THRESH).astype(np.uint8)

        img_means.append(im.mean() / 255.0)
        pos_frac.append(binm.mean())                       # fraction of cavity px

        # connected components = individual cavity blobs (label 0 is background)
        n_cc, _, cc_stats, _ = cv2.connectedComponentsWithStats(binm, connectivity=8)
        areas = cc_stats[1:, cv2.CC_STAT_AREA] if n_cc > 1 else np.array([])
        n_lesions.append(len(areas))
        lesion_areas.extend(areas.tolist())

    eda_stats[split] = dict(
        img_means=np.array(img_means),
        pos_frac=np.array(pos_frac),
        n_lesions=np.array(n_lesions),
        lesion_areas=np.array(lesion_areas, dtype=float),
    )
    print(f"{split:<6}: scanned {len(img_means)} patches")

print("Done. Stats stored in `eda_stats`.")


## 3 · Class imbalance — the headline finding

For each split we print the **overall cavity-pixel share** (pooled across all
patches) and the **average per-patch share**. Expect something around **2–3%**,
i.e. roughly a **1 : 30+** foreground-to-background ratio. That extreme imbalance
is exactly why pixel-accuracy is meaningless here (a model predicting "all
background" scores ~97%) and why the loss combines **focal** (down-weights easy
background) with **dice** (scores region overlap directly).


In [ ]:
# ---- 3. Class-imbalance summary ----------------------------------------
print(f"{'split':<8}{'overall cavity %':>18}{'mean per-patch %':>18}"
      f"{'empty masks':>13}{'bg:fg ratio':>14}")
print("-" * 71)
for split, st in eda_stats.items():
    pf = st["pos_frac"]
    overall = pf.mean() * 100                 # equal-size patches -> mean == pooled
    empty = int((pf == 0).sum())
    ratio = (1 - pf.mean()) / pf.mean() if pf.mean() > 0 else float("inf")
    print(f"{split:<8}{overall:>17.2f}%{overall:>17.2f}%{empty:>13}{ratio:>12.1f}:1")
print("-" * 71)
print("Takeaway: only a few % of pixels are cavity  ->  severe class imbalance")
print("          ->  motivates focal + dice loss instead of plain cross-entropy.")


## 4 · Distribution plots

Four views of the data distribution:

- **(a) Cavity-pixel fraction per patch** — how much of each patch is cavity.
  A right-skewed distribution piled up near zero visually confirms the imbalance.
- **(b) Mean image intensity per patch** — the brightness spread across X-rays;
  a wide spread means contrast normalization is worth considering.
- **(c) Number of cavities (connected components) per patch** — usually 1, but
  some patches contain several lesions (relevant because the demo app only boxes
  the largest one).
- **(d) Individual lesion area** (log scale) — how big cavities are in pixels;
  many are tiny, which is what makes them hard to segment.


In [ ]:
# ---- 4. Distribution plots ---------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
splits = list(eda_stats.keys())

# (a) cavity-pixel fraction per patch
ax = axes[0, 0]
for i, s in enumerate(splits):
    ax.hist(eda_stats[s]["pos_frac"] * 100, bins=60, alpha=0.6,
            color=EDA_PALETTE[i % len(EDA_PALETTE)], label=s)
ax.set_title("(a) Cavity-pixel fraction per patch")
ax.set_xlabel("% of patch that is cavity"); ax.set_ylabel("number of patches")
ax.legend()

# (b) mean image intensity per patch
ax = axes[0, 1]
for i, s in enumerate(splits):
    ax.hist(eda_stats[s]["img_means"], bins=50, alpha=0.6,
            color=EDA_PALETTE[i % len(EDA_PALETTE)], label=s)
ax.set_title("(b) Mean image intensity per patch")
ax.set_xlabel("mean pixel value (0-1)"); ax.set_ylabel("number of patches")
ax.legend()

# (c) lesions per patch
ax = axes[1, 0]
alln = np.concatenate([eda_stats[s]["n_lesions"] for s in splits])
maxn = int(alln.max()) if len(alln) else 1
bins = np.arange(0, maxn + 2) - 0.5
for i, s in enumerate(splits):
    ax.hist(eda_stats[s]["n_lesions"], bins=bins, alpha=0.6,
            color=EDA_PALETTE[i % len(EDA_PALETTE)], label=s)
ax.set_title("(c) Cavities (connected components) per patch")
ax.set_xlabel("# lesions in patch"); ax.set_ylabel("number of patches")
ax.set_xticks(range(0, maxn + 1)); ax.legend()

# (d) lesion area distribution (log-x)
ax = axes[1, 1]
for i, s in enumerate(splits):
    la = eda_stats[s]["lesion_areas"]
    la = la[la > 0]
    if len(la):
        ax.hist(la, bins=np.logspace(0, np.log10(la.max() + 1), 40), alpha=0.6,
                color=EDA_PALETTE[i % len(EDA_PALETTE)], label=s)
ax.set_xscale("log")
ax.set_title("(d) Individual lesion area")
ax.set_xlabel("lesion area (pixels, log scale)"); ax.set_ylabel("count")
ax.legend()

plt.tight_layout(); plt.show()


## 5 · Where do cavities appear? (spatial heatmap)

By averaging every binary mask in the training set we get a **spatial prior**: a
heatmap of how often each pixel location is labelled cavity. If lesions are spread
across the patch the map is diffuse; a hot centre would mean patches were cropped
tightly around the lesion (useful to know, because it affects how the model
generalizes to full radiographs).


In [ ]:
# ---- 5. Average training mask = cavity location heatmap -----------------
train_img_dir, train_msk_dir = EDA_SPLITS["train"]
files = eda_list_pngs(train_msk_dir)

acc, n = None, 0
for f in files:
    mk = cv2.imread(os.path.join(train_msk_dir, f), cv2.IMREAD_GRAYSCALE)
    if mk is None:
        continue
    binm = (mk > EDA_POS_THRESH).astype(np.float32)
    acc = binm if acc is None else acc + binm
    n += 1

heat = acc / max(n, 1)
plt.figure(figsize=(5.2, 4.4))
plt.imshow(heat, cmap="inferno")
plt.colorbar(label="P(pixel is cavity)")
plt.title(f"Cavity spatial prior  (avg of {n} train masks)")
plt.axis("off"); plt.tight_layout(); plt.show()

print(f"Hottest pixel probability: {heat.max():.3f}  |  mean: {heat.mean():.4f}")


## 6 · Visual samples — image / mask / overlay

Numbers only go so far; here we eyeball a handful of patches with a decent amount
of cavity (2–10% of the patch) so the labels are clearly visible. Each row shows
the raw X-ray, its ground-truth mask, and the mask tinted red over the image —
a quick sanity check that annotations line up with visible lesions.


In [ ]:
# ---- 6. Sample image / mask / overlay grid -----------------------------
img_dir, msk_dir = EDA_SPLITS["val"]         # use val for a clean peek
files = eda_list_pngs(img_dir)

# pick patches with a visible cavity, preferring different source X-rays
picks, seen_src = [], set()
for f in files:
    mk = cv2.imread(os.path.join(msk_dir, f), cv2.IMREAD_GRAYSCALE)
    if mk is None:
        continue
    frac = (mk > EDA_POS_THRESH).mean()
    src = f.split("_jpg")[0].split(".rf.")[0]   # coarse source-image id
    if 0.02 < frac < 0.10 and src not in seen_src:
        picks.append(f); seen_src.add(src)
    if len(picks) >= 5:
        break

rows = len(picks)
fig, axes = plt.subplots(rows, 3, figsize=(9, 3 * rows))
if rows == 1:
    axes = axes[None, :]
for r, f in enumerate(picks):
    im = cv2.imread(os.path.join(img_dir, f), cv2.IMREAD_GRAYSCALE)
    mk = cv2.imread(os.path.join(msk_dir, f), cv2.IMREAD_GRAYSCALE)
    binm = (mk > EDA_POS_THRESH)
    overlay = np.stack([im] * 3, axis=-1).astype(float)
    overlay[binm] = overlay[binm] * 0.35 + np.array([220, 50, 50]) * 0.65

    axes[r, 0].imshow(im, cmap="gray");            axes[r, 0].set_title("X-ray patch")
    axes[r, 1].imshow(mk, cmap="gray");            axes[r, 1].set_title("cavity mask")
    axes[r, 2].imshow(overlay.astype(np.uint8));   axes[r, 2].set_title("overlay")
    for c in range(3):
        axes[r, c].axis("off")
plt.tight_layout(); plt.show()
print("Sampled files:", picks)


## 7 · EDA summary

Key facts to carry into modelling / the presentation:

- **~5,000 paired `128×128` grayscale patches**, split train/val with **no
  filename leakage**; every image has a matching mask.
- Masks are effectively **binary** (`0` / `255`).
- **Severe class imbalance (~2–3% cavity pixels)** — the defining property of this
  dataset and the direct justification for the **focal + dice** loss and for
  reporting **Dice / precision / recall** instead of pixel accuracy.
- Most patches contain **one** lesion, but some contain several, and many lesions
  are **small** — which is what makes the segmentation genuinely hard.

Natural next steps this EDA points to: patient-level (source-image) splitting to
rule out leakage between patches of the same X-ray, adding **cavity-free
patches** as negatives, and trying contrast normalization (CLAHE) given the
spread in image intensity.


In [ ]:
# ---- 7. Compact numeric summary ----------------------------------------
print("DATASET EDA SUMMARY")
print("=" * 52)
for split, v in eda_inventory.items():
    st = eda_stats.get(split, {})
    pf = st.get("pos_frac", np.array([]))
    print(f"{split.upper():<6}  pairs={v['paired']:>5}   "
          f"cavity%={pf.mean()*100 if len(pf) else 0:5.2f}   "
          f"empty_masks={int((pf==0).sum()) if len(pf) else 0}")
print("=" * 52)
print("All patches 128x128 grayscale | masks binary | see plots above.")


# ============================================================
# TRAINING — U-Net cavity segmentation
# ============================================================
The cell that used to hold everything is split below so each stage prints its
own result. The logic is unchanged from the original pipeline.


### T1 · Imports, paths & hyper-parameters

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models

# --- dataset locations ---
TRAIN_IMG_DIR = "/content/drive/MyDrive/teeth/patch_dataset/train/images"
TRAIN_MSK_DIR = "/content/drive/MyDrive/teeth/patch_dataset/train/masks"
VAL_IMG_DIR   = "/content/drive/MyDrive/teeth/patch_dataset/val/images"
VAL_MSK_DIR   = "/content/drive/MyDrive/teeth/patch_dataset/val/masks"

# --- where the trained models are written ---
SAVE_BEST  = "/content/drive/MyDrive/teeth/unet_best_model.h5"
SAVE_FINAL = "/content/drive/MyDrive/teeth/unet_cavity_final.h5"

# --- hyper-parameters ---
IMG_SIZE   = 128
BATCH_SIZE = 16
EPOCHS     = 100
LR         = 1e-4

print("Config ready | IMG_SIZE", IMG_SIZE, "| BATCH", BATCH_SIZE,
      "| EPOCHS", EPOCHS, "| LR", LR)

### T2 · Data-loading function

In [ ]:
def load_data(img_dir, mask_dir):
    images = []
    masks = []

    files = sorted(os.listdir(img_dir))

    for fname in files:
        img_path = os.path.join(img_dir, fname)
        msk_path = os.path.join(mask_dir, fname)

        if not os.path.exists(msk_path):
            print("Mask missing for:", fname)
            continue

        img = cv2.imread(img_path, 0)
        msk = cv2.imread(msk_path, 0)

        if img is None or msk is None:
            print("Read error for:", fname)
            continue

        # Resize
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        msk = cv2.resize(msk, (IMG_SIZE, IMG_SIZE))

        # Normalize
        img = img / 255.0
        msk = msk / 255.0
        msk = (msk > 0.5).astype(np.float32)

        img = img[..., np.newaxis]
        msk = msk[..., np.newaxis]

        images.append(img)
        masks.append(msk)

    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.float32)

print("load_data() defined.")

### T3 · Load the train & validation sets into memory

In [ ]:
print("Loading training data...")
X_train, y_train = load_data(TRAIN_IMG_DIR, TRAIN_MSK_DIR)

print("Loading validation data...")
X_val, y_val = load_data(VAL_IMG_DIR, VAL_MSK_DIR)

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)

### T4 · Augmentation (random flips, re-rolled every epoch)

In [ ]:
def augment_data(images, masks):
    for i in range(len(images)):
        if np.random.rand() < 0.5:
            images[i] = np.fliplr(images[i])
            masks[i]  = np.fliplr(masks[i])

        if np.random.rand() < 0.5:
            images[i] = np.flipud(images[i])
            masks[i]  = np.flipud(masks[i])

    return images, masks

print("augment_data() defined.")

### T5 · U-Net architecture

In [ ]:
def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 1)):
    inputs = layers.Input(input_shape)

    # Encoder
    c1 = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
    c1 = layers.Conv2D(32, 3, activation="relu", padding="same")(c1)
    p1 = layers.MaxPooling2D()(c1)

    c2 = layers.Conv2D(64, 3, activation="relu", padding="same")(p1)
    c2 = layers.Conv2D(64, 3, activation="relu", padding="same")(c2)
    p2 = layers.MaxPooling2D()(c2)

    c3 = layers.Conv2D(128, 3, activation="relu", padding="same")(p2)
    c3 = layers.Conv2D(128, 3, activation="relu", padding="same")(c3)
    p3 = layers.MaxPooling2D()(c3)

    # Bottleneck
    c4 = layers.Conv2D(256, 3, activation="relu", padding="same")(p3)
    c4 = layers.Conv2D(256, 3, activation="relu", padding="same")(c4)

    # Decoder
    u5 = layers.UpSampling2D()(c4)
    u5 = layers.Concatenate()([u5, c3])
    c5 = layers.Conv2D(128, 3, activation="relu", padding="same")(u5)
    c5 = layers.Conv2D(128, 3, activation="relu", padding="same")(c5)

    u6 = layers.UpSampling2D()(c5)
    u6 = layers.Concatenate()([u6, c2])
    c6 = layers.Conv2D(64, 3, activation="relu", padding="same")(u6)
    c6 = layers.Conv2D(64, 3, activation="relu", padding="same")(c6)

    u7 = layers.UpSampling2D()(c6)
    u7 = layers.Concatenate()([u7, c1])
    c7 = layers.Conv2D(32, 3, activation="relu", padding="same")(u7)
    c7 = layers.Conv2D(32, 3, activation="relu", padding="same")(c7)

    outputs = layers.Conv2D(1, 1, activation="sigmoid")(c7)

    return models.Model(inputs, outputs)

print("build_unet() defined.")

### T6 · Loss & metric functions (focal + dice)

In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def focal_loss(gamma=2., alpha=0.25):
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        return -tf.reduce_mean(alpha * tf.pow(1. - pt, gamma) * tf.math.log(pt))
    return loss

def focal_dice_loss(y_true, y_pred):
    return focal_loss()(y_true, y_pred) + (1 - dice_coef(y_true, y_pred))

print("Loss / metric functions defined.")

### T7 · Build & compile the model

In [ ]:
model = build_unet()

model.compile(
    optimizer=tf.keras.optimizers.Adam(LR),
    loss=focal_dice_loss,
    metrics=[
        dice_coef,
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)

model.summary()

### T8 · Callbacks (checkpoint, early-stop, LR schedule)

In [ ]:
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    SAVE_BEST,
    monitor="val_dice_coef",
    save_best_only=True,
    mode="max",
    verbose=0
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_dice_coef",
    patience=15,
    mode="max",
    restore_best_weights=True,
    verbose=0
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_dice_coef",
    factor=0.5,
    patience=7,
    mode="max",
    verbose=0
)

print("Callbacks ready.")

### T9 · Train
The per-epoch loop re-rolls the flip augmentation each epoch. Metrics are also
collected into `training_history` so the evaluation cells can plot the learning
curves (this only records numbers — it does not change training).

In [ ]:
training_history = {}   # accumulates metrics across epochs for later plots

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    X_train_aug, y_train_aug = augment_data(X_train.copy(), y_train.copy())

    history = model.fit(
        X_train_aug, y_train_aug,
        validation_data=(X_val, y_val),
        epochs=1,
        batch_size=BATCH_SIZE,
        callbacks=[checkpoint, early_stop, reduce_lr],
        verbose=1
    )

    # record this epoch's metrics (purely for later plotting)
    for k, v in history.history.items():
        training_history.setdefault(k, []).extend(v)

    if len(history.history["val_dice_coef"]) == 0:
        break

### T10 · Save the trained model

In [ ]:
model.save(SAVE_FINAL)

print("\nTraining completed successfully.")
print("Best model saved at :", SAVE_BEST)
print("Final model saved at:", SAVE_FINAL)

# ============================================================
# EVALUATION — how good is the trained model?
# ============================================================
These cells run *after* training and score the model on the **validation set**
(`X_val`, `y_val`). For segmentation with heavy class imbalance the meaningful
metrics are **Dice**, **IoU (Jaccard)**, **precision**, **recall** and **F1** —
not pixel accuracy. We also sweep the decision threshold, show a pixel-level
confusion matrix, plot the training curves, and look at sample predictions.

### E1 · Predict on the validation set & define metric helpers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# raw per-pixel probabilities from the trained model
y_prob = model.predict(X_val, batch_size=BATCH_SIZE, verbose=1)
y_true = y_val.astype(np.float32)

def seg_scores(y_true, y_pred_bin, eps=1e-6):
    """Pixel-level segmentation scores for a binary prediction."""
    yt = y_true.reshape(-1).astype(bool)
    yp = y_pred_bin.reshape(-1).astype(bool)
    tp = int(np.logical_and(yt,  yp).sum())
    fp = int(np.logical_and(~yt, yp).sum())
    fn = int(np.logical_and(yt, ~yp).sum())
    tn = int(np.logical_and(~yt, ~yp).sum())
    dice = (2*tp + eps) / (2*tp + fp + fn + eps)
    iou  = (tp + eps)   / (tp + fp + fn + eps)
    prec = (tp + eps)   / (tp + fp + eps)
    rec  = (tp + eps)   / (tp + fn + eps)
    f1   = (2*prec*rec) / (prec + rec + eps)
    acc  = (tp + tn)    / (tp + tn + fp + fn + eps)
    spec = (tn + eps)   / (tn + fp + eps)
    return dict(dice=dice, iou=iou, precision=prec, recall=rec, f1=f1,
                pixel_acc=acc, specificity=spec, tp=tp, fp=fp, fn=fn, tn=tn)

print("Predictions ready:", y_prob.shape)

### E2 · Headline metrics @ 0.5  +  best-threshold sweep
`Dice` and `IoU` are the standard overlap scores. Precision high / recall lower
means the model is *conservative* — it rarely false-alarms but misses some cavity
pixels. The sweep finds the threshold that maximises Dice (useful because 0.5 is
rarely optimal for imbalanced masks).

In [ ]:
m05 = seg_scores(y_true, (y_prob > 0.5))

print("VALIDATION METRICS  @ threshold 0.5")
print("-" * 40)
for k in ["dice", "iou", "precision", "recall", "f1", "specificity", "pixel_acc"]:
    print(f"  {k:<12}: {m05[k]:.4f}")
print(f"  (pixel accuracy is ~{m05['pixel_acc']*100:.1f}% — high only because "
      f"most pixels are background)")

# threshold sweep
thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)
sweep = [(float(t), seg_scores(y_true, (y_prob > t))) for t in thresholds]
best_t, best_m = max(sweep, key=lambda tm: tm[1]["dice"])

print("\nThreshold sweep (Dice / IoU / Precision / Recall):")
for t, mm in sweep:
    star = "  <== best Dice" if t == best_t else ""
    print(f"  thr={t:.2f} | Dice {mm['dice']:.3f} | IoU {mm['iou']:.3f} | "
          f"P {mm['precision']:.3f} | R {mm['recall']:.3f}{star}")

print(f"\nBest threshold = {best_t:.2f}  ->  Dice {best_m['dice']:.4f}, "
      f"IoU {best_m['iou']:.4f}, P {best_m['precision']:.4f}, R {best_m['recall']:.4f}")

### E3 · Image-level detection rate
The end product answers *"does this patch contain a cavity?"*. A patch counts as
**detected** if the predicted mask has at least `MIN_AREA` positive pixels (the
same noise filter the demo app uses). Since every validation patch actually
contains a cavity, this is the model's **sensitivity** at the patch level.

In [ ]:
MIN_AREA = 20   # predicted positive pixels required to call a patch "cavity"

pred_bin = (y_prob > 0.5).astype(np.uint8)
pred_area = pred_bin.reshape(len(pred_bin), -1).sum(axis=1)   # px per patch
true_area = (y_true > 0.5).reshape(len(y_true), -1).sum(axis=1)

has_true = true_area > 0
has_pred = pred_area >= MIN_AREA

detected = int(np.logical_and(has_true, has_pred).sum())
missed   = int(np.logical_and(has_true, ~has_pred).sum())

print(f"Validation patches containing a cavity : {int(has_true.sum())}")
print(f"  correctly flagged (detected)         : {detected}")
print(f"  missed (no/low prediction)           : {missed}")
print(f"  patch-level sensitivity              : {detected/max(int(has_true.sum()),1)*100:.1f}%")
print("\nNote: this dataset has no cavity-free patches, so specificity / "
      "false-positive rate cannot be measured here — adding negative patches is "
      "a key next step.")

### E4 · Pixel confusion matrix & threshold curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))

# --- (left) confusion matrix at 0.5 (in % of all pixels) ---
cm = np.array([[m05["tn"], m05["fp"]],
               [m05["fn"], m05["tp"]]], dtype=float)
cm_pct = cm / cm.sum() * 100
ax = axes[0]
im = ax.imshow(cm_pct, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["pred bg", "pred cavity"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["true bg", "true cavity"])
ax.set_title("Pixel confusion matrix @ 0.5  (% of all pixels)")
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm_pct[i,j]:.2f}%", ha="center", va="center",
                color="white" if cm_pct[i,j] > cm_pct.max()/2 else "black",
                fontsize=12)
ax.grid(False)

# --- (right) Dice / precision / recall vs threshold ---
ts   = [t for t, _ in sweep]
dice = [mm["dice"]      for _, mm in sweep]
prec = [mm["precision"] for _, mm in sweep]
rec  = [mm["recall"]    for _, mm in sweep]
ax = axes[1]
ax.plot(ts, dice, "-o", color="#1E3A8F", label="Dice")
ax.plot(ts, prec, "-o", color="#8FAE9C", label="Precision")
ax.plot(ts, rec,  "-o", color="#C4453B", label="Recall")
ax.axvline(best_t, ls="--", color="grey", alpha=0.7)
ax.set_title("Metrics vs decision threshold")
ax.set_xlabel("threshold"); ax.set_ylabel("score"); ax.legend()

plt.tight_layout(); plt.show()

### E5 · Training curves & over-fitting check
Train vs validation Dice over the epochs. A small, stable gap = healthy fit;
a widening gap (train ≫ val) would signal over-fitting.

In [ ]:
if "training_history" in globals() and training_history.get("dice_coef"):
    h = training_history
    ep = range(1, len(h["dice_coef"]) + 1)

    fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
    ax[0].plot(ep, h["dice_coef"], color="#1E3A8F", label="train Dice")
    if "val_dice_coef" in h:
        ax[0].plot(ep, h["val_dice_coef"], color="#C4453B", label="val Dice")
    ax[0].set_title("Dice vs epoch"); ax[0].set_xlabel("epoch")
    ax[0].set_ylabel("Dice"); ax[0].legend()

    ax[1].plot(ep, h["loss"], color="#1E3A8F", label="train loss")
    if "val_loss" in h:
        ax[1].plot(ep, h["val_loss"], color="#C4453B", label="val loss")
    ax[1].set_title("Loss vs epoch"); ax[1].set_xlabel("epoch")
    ax[1].set_ylabel("focal + dice loss"); ax[1].legend()
    plt.tight_layout(); plt.show()

    if "val_dice_coef" in h:
        gap = h["dice_coef"][-1] - h["val_dice_coef"][-1]
        print(f"Final train Dice {h['dice_coef'][-1]:.3f} | "
              f"val Dice {h['val_dice_coef'][-1]:.3f} | gap {gap:+.3f}")
else:
    print("No training_history found — run the training loop (cell T9) first.")

### E6 · Sample predictions — image / ground truth / prediction

In [ ]:
# show a few val patches that actually contain a cavity
idx = np.where(true_area > 30)[0][:5]

rows = len(idx)
fig, axes = plt.subplots(rows, 3, figsize=(9, 3 * rows))
if rows == 1:
    axes = axes[None, :]
for r, i in enumerate(idx):
    img = X_val[i, ..., 0]
    gt  = y_true[i, ..., 0]
    pr  = (y_prob[i, ..., 0] > 0.5).astype(np.uint8)

    axes[r, 0].imshow(img, cmap="gray");  axes[r, 0].set_title("X-ray patch")
    axes[r, 1].imshow(gt,  cmap="gray");  axes[r, 1].set_title("ground truth")
    axes[r, 2].imshow(pr,  cmap="gray");  axes[r, 2].set_title("prediction @0.5")
    for c in range(3):
        axes[r, c].axis("off")
plt.tight_layout(); plt.show()

print(f"Final validation summary: Dice {m05['dice']:.3f} | IoU {m05['iou']:.3f} | "
      f"P {m05['precision']:.3f} | R {m05['recall']:.3f} | F1 {m05['f1']:.3f}")